# Notebook 05 -- Selección de Expertos

---


## Objetivo

Seleccionar e identificar los mejores modelos para cada dominio (cluster). Evaluar 5 familias de modelos (5 expertos/familia) con walk-forward validation. Comparar enfoques homogéneo vs. heterogéneo.


## Entrada

data/processed/features.parquet + asignaciones de cluster + clasificadores entrenados


## Salida

Modelos expertos optimizados (5 familias × 5 expertos = 25 modelos) + tabla de desempeño por experto


## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

## Imports y Configuración

In [ ]:
import os
import warnings
import pickle
from pathlib import Path
from copy import deepcopy
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet, BayesianRidge, SGDRegressor,
)
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
RANDOM_SEED = 42
N_JOBS = -1

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ── Configuración de gráficos ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.figsize": (12, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "font.family": "serif",
    "axes.grid": True,
    "grid.alpha": 0.3,
})
PALETTE = sns.color_palette("Set2", 8)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo PyTorch: {DEVICE}")
print(f"Semilla aleatoria:   {RANDOM_SEED}")

## Cargar Datos

In [ ]:
# ── Cargar variables y asignaciones de clúster ───────────────────────────────────────
features_path = os.path.join(DATA_DIR, "processed", "features.parquet")
clusters_path = os.path.join(DATA_DIR, "clustering", "cluster_assignments.parquet")

df_feat = pd.read_parquet(features_path)
df_clust = pd.read_parquet(clusters_path)

print(f"Forma de variables:                 {df_feat.shape}")
print(f"Forma de asignaciones de clúster:  {df_clust.shape}")
print(f"\nColumnas de variables ({len(df_feat.columns)}):")
print(df_feat.columns.tolist()[:20], "..." if len(df_feat.columns) > 20 else "")
print(f"\nColumnas de clúster: {df_clust.columns.tolist()}")
display(df_clust.head())

In [ ]:
df_feat.head()

In [ ]:
df_feat = df_feat[~df_feat.index.year.isin([2020, 2021])]

In [ ]:
df_clust = pd.read_parquet(clusters_path)

# Asegurar que 'datetime' en df_feat sea una columna y de tipo datetime
df_feat = df_feat.reset_index() # Convertir el índice datetime en una columna
df_feat["datetime"] = pd.to_datetime(df_feat["datetime"]) # Asegurar que sea de tipo datetime

# Determinar la columna que se usará para hacer el merge. Debe ser 'datetime' para df_feat
feat_key = "datetime"

# Asegurar que 'expert_id' esté correctamente formado si aún no existe.
if "expert_id" not in df_clust.columns:
    td_col = [c for c in df_clust.columns if "type_day" in c.lower() or "cluster_day" in c.lower()]
    tw_col = [c for c in df_clust.columns if "type_week" in c.lower() or "cluster_week" in c.lower()]

    if td_col and tw_col:
        df_clust["expert_id"] = df_clust[td_col[0]].astype(str) + "_" + df_clust[tw_col[0]].astype(str)
    elif td_col:
        df_clust["expert_id"] = df_clust[td_col[0]].astype(str)
    else:
        # Respaldo si no se encuentran columnas específicas de clúster
        print("ADVERTENCIA: No se pudieron encontrar 'cluster_day' o 'cluster_week' para formar expert_id. Se usará un valor por defecto.")
        df_clust["expert_id"] = "default_expert"


# Realizar un merge por la izquierda para conservar todas las filas de df_feat y agregar 'expert_id' desde df_clust
df = df_feat.merge(
    df_clust[[feat_key, "expert_id"]],
    on=feat_key,
    how="left", # Usar merge por la izquierda para preservar todas las filas de df_feat
)

if df['expert_id'].isnull().any():
    print("ADVERTENCIA: Algunas filas de df_feat no pudieron asociarse con expert_id. Se rellenarán los NaN.")
    original_shape = df.shape[0]
    df.dropna(subset=['expert_id'], inplace=True)
    if df.shape[0] < original_shape:
        print(f"Se eliminaron {original_shape - df.shape[0]} filas debido a expert_id faltante después del merge.")

df = df.sort_values(feat_key).reset_index(drop=True)

print(f"Forma después del merge: {df.shape}")
if not df.empty:
    print(f"Expert IDs:   {sorted(df['expert_id'].unique())}")
    print(f"\nObservaciones por experto:")
    print(df["expert_id"].value_counts().sort_index())
else:
    print("ADVERTENCIA: El DataFrame final combinado 'df' sigue vacío.")

In [ ]:
df_clust.head()

In [ ]:
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df_feat.columns)

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
# ── Identificar columna objetivo y columnas de variables ──────────────────────────────────────
TARGET = None
for candidate in ["demand", "demanda", "target", "y", "load", "demand_kwh"]:
    if candidate in df.columns:
        TARGET = candidate
        break
if TARGET is None:
    # Heurística: columna numérica más probable de ser el objetivo
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    TARGET = numeric_cols[0]
    print(f"ADVERTENCIA: La variable objetivo se detectó automáticamente como '{TARGET}'. Verifica que sea correcta.")

EXCLUDE_COLS = {feat_key, TARGET, "expert_id"}
#FEATURE_COLS = [
#    c for c in df.columns
#    if c not in EXCLUDE_COLS and df[c].dtype in [np.float64, np.float32, np.int64, np.int32]
#]

FEATURE_COLS = calendar_lags

print(f"Columna objetivo:   {TARGET}")
print(f"Columnas de variables: {len(FEATURE_COLS)}")
print(f"Variables de ejemplo: {FEATURE_COLS}")

---

## Metricas

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Métricas — 100 % autocontenidas (sin importaciones externas)
# ══════════════════════════════════════════════════════════════════════════════

def calc_rmse(y_true, y_pred):
    """Raíz del Error Cuadrático Medio."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def calc_mae(y_true, y_pred):
    """Error Absoluto Medio."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def calc_mape(y_true, y_pred):
    """Error Porcentual Absoluto Medio."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return float("inf")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])))


def calc_mase(y_true, y_pred, seasonal_period=168):
    """Error Absoluto Medio Escalado (línea base ingenua estacional = 168h = 1 semana)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    naive_errors = np.abs(y_true[seasonal_period:] - y_true[:-seasonal_period])
    naive_mae = np.mean(naive_errors) if len(naive_errors) > 0 else 0.0
    if naive_mae == 0:
        return float("inf")
    forecast_mae = np.mean(np.abs(y_true - y_pred))
    return float(forecast_mae / naive_mae)


def calc_all_metrics(y_true, y_pred):
    """Retorna un diccionario con RMSE, MAE, MAPE, MASE."""
    return {
        "rmse": calc_rmse(y_true, y_pred),
        "mae": calc_mae(y_true, y_pred),
        "mape": calc_mape(y_true, y_pred),
        "mase": calc_mase(y_true, y_pred),
    }


def weighted_mase_global(mase_per_expert, n_per_expert):
    """MASE_global = sum(n_k / N * MASE_k) ponderado por el tamaño del clúster."""
    total_n = sum(n_per_expert.values())
    if total_n == 0:
        return float("inf")
    return sum(
        (n_per_expert[k] / total_n) * mase_per_expert[k]
        for k in mase_per_expert
    )


print("Funciones de métricas definidas: calc_rmse, calc_mae, calc_mape, calc_mase, calc_all_metrics, weighted_mase_global")

---

## Definición de Familias de Modelos (5 por familia)

In [ ]:
class _DLModule(nn.Module):
    """Módulo multiarquitectura para pronóstico tabular (ANN, RNN, TCN, Transformer)."""
    def __init__(self, input_size, hidden_size, num_layers, arch_type="lstm", dropout=0.1):
        super().__init__()
        self.arch_type = arch_type

        if arch_type in ["lstm", "gru"]:
            RNNClass = nn.LSTM if arch_type == "lstm" else nn.GRU
            self.net = RNNClass(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            self.fc = nn.Linear(hidden_size, 1)

        elif arch_type == "ann":
            layers = []
            curr_size = input_size
            for _ in range(num_layers):
                layers.extend([nn.Linear(curr_size, hidden_size), nn.ReLU(), nn.Dropout(dropout)])
                curr_size = hidden_size
            layers.append(nn.Linear(hidden_size, 1))
            self.net = nn.Sequential(*layers)

        elif arch_type == "tcn":
            self.net = nn.Sequential(
                nn.Conv1d(1, hidden_size, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1)
            )
            self.fc = nn.Linear(hidden_size, 1)

        elif arch_type == "transformer":
            encoder_layer = nn.TransformerEncoderLayer(d_model=input_size, nhead=1, dim_feedforward=hidden_size, batch_first=True)
            self.net = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.fc = nn.Linear(input_size, 1)

    def forward(self, x):
        if self.arch_type in ["lstm", "gru"]:
            out, _ = self.net(x)
            return self.fc(out[:, -1, :]).squeeze(-1)
        elif self.arch_type == "ann":
            return self.net(x.squeeze(1)).squeeze(-1)
        elif self.arch_type == "tcn":
            out = self.net(x)
            return self.fc(out.squeeze(-1)).squeeze(-1)
        elif self.arch_type == "transformer":
            out = self.net(x)
            return self.fc(out[:, -1, :]).squeeze(-1)

class TorchRNNRegressor(BaseEstimator, RegressorMixin):
    """Wrapper compatible con sklearn alrededor de modelos PyTorch para regresión."""
    def __init__(self, hidden_size=64, num_layers=2, arch_type="lstm", lr=1e-3, epochs=50, batch_size=256, patience=5, dropout=0.1, random_state=42):
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.arch_type = arch_type
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.dropout = dropout
        self.random_state = random_state

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        X, y = np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)
        self.x_mean_, self.x_std_ = X.mean(axis=0), X.std(axis=0) + 1e-8
        self.y_mean_, self.y_std_ = y.mean(), y.std() + 1e-8
        X_t = torch.tensor((X - self.x_mean_) / self.x_std_).unsqueeze(1).to(DEVICE)
        y_t = torch.tensor((y - self.y_mean_) / self.y_std_).to(DEVICE)
        n_val = max(1, int(0.15 * len(X_t)))
        train_ds = TensorDataset(X_t[:-n_val], y_t[:-n_val])
        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=False)
        self.model_ = _DLModule(X.shape[1], self.hidden_size, self.num_layers, self.arch_type, self.dropout).to(DEVICE)
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=self.lr)
        criterion = nn.MSELoss()
        best_val_loss, patience_counter, best_state = float("inf"), 0, None
        for epoch in range(self.epochs):
            self.model_.train()
            for xb, yb in train_loader:
                optimizer.zero_grad(); pred = self.model_(xb); loss = criterion(pred, yb); loss.backward(); optimizer.step()
            self.model_.eval()
            with torch.no_grad():
                val_loss = criterion(self.model_(X_t[-n_val:]), y_t[-n_val:]).item()
            if val_loss < best_val_loss:
                best_val_loss, patience_counter, best_state = val_loss, 0, deepcopy(self.model_.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= self.patience: break
        if best_state: self.model_.load_state_dict(best_state)
        return self

    def predict(self, X):
        X_t = torch.tensor((np.asarray(X, dtype=np.float32) - self.x_mean_) / self.x_std_).unsqueeze(1).to(DEVICE)
        self.model_.eval()
        with torch.no_grad(): return self.model_(X_t).cpu().numpy() * self.y_std_ + self.y_mean_

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return float(1 - ss_res / ss_tot) if ss_tot != 0 else 0.0

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4b. Catálogo de modelos — 5 modelos por familia
# ══════════════════════════════════════════════════════════════════════════════

def get_linear_models():
    """Retorna un diccionario con 5 modelos lineales (cada uno envuelto en un Pipeline con StandardScaler)."""
    return {
        "ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0, random_state=RANDOM_SEED)),
        ]),
        "lasso": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_SEED)),
        ]),
        "elasticnet": Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_SEED)),
        ]),
        "bayesian_ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", BayesianRidge()),
        ]),
        "sgd": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SGDRegressor(penalty="elasticnet", max_iter=2000, random_state=RANDOM_SEED)),
        ]),
    }


def get_ml_models():
    """Retorna un diccionario con 5 modelos de ML (ensambles basados en árboles)."""
    return {
        "lgbm": lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=31,
            verbose=-1, random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "xgboost": xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            verbosity=0, random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "random_forest": RandomForestRegressor(
            n_estimators=200, max_depth=None,
            random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "extra_trees": ExtraTreesRegressor(
            n_estimators=200, max_depth=None,
            random_state=RANDOM_SEED, n_jobs=N_JOBS,
        ),
        "gradient_boosting": GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=5,
            random_state=RANDOM_SEED,
        ),
    }

def get_dl_models():
    """Retorna un diccionario con 5 modelos DL incluyendo LSTM, GRU, ANN, TCN y Transformer."""
    return {
        "lstm_standard": TorchRNNRegressor(
            hidden_size=64, num_layers=2, arch_type="lstm",
            random_state=RANDOM_SEED),
        "gru_standard": TorchRNNRegressor(
            hidden_size=64, num_layers=2, arch_type="gru",
            random_state=RANDOM_SEED),
        "ann_dense": TorchRNNRegressor(
            hidden_size=128, num_layers=3, arch_type="ann",
            random_state=RANDOM_SEED),
        "tcn_conv": TorchRNNRegressor(
            hidden_size=64, arch_type="tcn",
            random_state=RANDOM_SEED),
        "transformer_lite": TorchRNNRegressor(
            hidden_size=64, num_layers=2, arch_type="transformer",
            random_state=RANDOM_SEED),
    }

# Resumen
print("Familias de modelos definidas:")
print(f"  linear : {list(get_linear_models().keys())}")
print(f"  ml     : {list(get_ml_models().keys())}")
print(f"  dl     : {list(get_dl_models().keys())}")

---

## Motor de Evaluación Walk-Forward

In [ ]:
# ───────────────────────────────────────────────────────────
# Motor de validación cruzada walk-forward
# ─────────────────────────────────────────────────────────────

def walk_forward_evaluate(model, X, y, n_splits=3, gap=0):
    """Evalúa un modelo usando validación cruzada walk-forward (ventana expansiva).

    Parámetros
    ----------
    model : estimador compatible con sklearn
    X : np.ndarray, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    n_splits : int — número de particiones de validación cruzada
    gap : int — separación (en horas) entre entrenamiento y prueba para evitar leakage

    Retorna
    -------
    dict con claves: rmse, mae, mape, mase (promediados entre folds),
         además de la lista fold_metrics.
    """
    n_samples = len(X)
    effective_n_splits = n_splits

    # Ajustar dinámicamente n_splits para asegurar que TimeSeriesSplit pueda crearse
    # TimeSeriesSplit requiere n_splits >= 2.
    tscv = None  # Inicializar tscv fuera del bucle
    while effective_n_splits >= 2:
        try:
            tscv_candidate = TimeSeriesSplit(n_splits=effective_n_splits, gap=gap)
            first_split = next(iter(tscv_candidate.split(X)), None)
            if first_split is not None:  # Si se generó correctamente una primera partición
                tscv = tscv_candidate
                break  # Se encontró una configuración válida de TimeSeriesSplit
            else:
                # No se pudieron generar particiones con esta configuración
                effective_n_splits -= 1
        except ValueError as e:
            # El constructor de TimeSeriesSplit puede lanzar ValueError si los parámetros son imposibles
            effective_n_splits -= 1
        except Exception as e:
            print(f"    [ADVERTENCIA] Error inesperado durante la creación/verificación de TimeSeriesSplit para n_splits={effective_n_splits}: {e}")
            effective_n_splits -= 1

    if tscv is None:  # Si el bucle terminó y no se encontró un tscv válido
        print(f"    [ADVERTENCIA] Datos demasiado pequeños para TimeSeriesSplit (n_samples={n_samples}, gap={gap}, no fue posible encontrar >=2 particiones válidas). Se omitirá la validación cruzada.")
        return {
            "rmse": float("inf"), "mae": float("inf"),
            "mape": float("inf"), "mase": float("inf"),
            "fold_metrics": [],
        }

    fold_metrics = []

    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Verificación básica para asegurar que los conjuntos de entrenamiento/prueba no estén vacíos
        if len(X_train) == 0 or len(X_test) == 0:
            print(f"    [ADVERTENCIA] El fold {fold_idx} produjo un conjunto de entrenamiento o prueba vacío. Se omitirá este fold.")
            metrics = {
                "fold": fold_idx,
                "rmse": float("inf"),
                "mae": float("inf"),
                "mape": float("inf"),
                "mase": float("inf"),
            }
        else:
            try:
                m = deepcopy(model)
                m.fit(X_train, y_train)
                y_pred = m.predict(X_test)
                metrics = calc_all_metrics(y_test, y_pred)
                metrics["fold"] = fold_idx
            except Exception as e:
                print(f"    [ADVERTENCIA] El fold {fold_idx} falló durante el ajuste/predicción del modelo: {e}")
                metrics = {
                    "fold": fold_idx,
                    "rmse": float("inf"),
                    "mae": float("inf"),
                    "mape": float("inf"),
                    "mase": float("inf"),
                }
        fold_metrics.append(metrics)

    # Promedio entre folds
    avg = {}
    for metric in ["rmse", "mae", "mape", "mase"]:
        vals = [fm[metric] for fm in fold_metrics if np.isfinite(fm[metric])]
        avg[metric] = float(np.mean(vals)) if vals else float("inf")

    avg["fold_metrics"] = fold_metrics
    return avg


print("Motor de evaluación walk-forward listo (TimeSeriesSplit, n_splits=3, gap=0).")

---

## 1. Restricciones Bias-Varianza por Clúster

Antes de ejecutar las evaluaciones, aplicamos el principio de bias-varianza para restringir la
complejidad del modelo según el tamaño del clúster:

| Tamaño del clúster | Familias permitidas | Justificación |
|:---|:---|:---|
| $n < 50$ | Solo Ridge, Lasso | Muy pocas observaciones — se requiere máxima regularización para evitar overfitting. Los ensambles de árboles tienen demasiados grados de libertad. |
| $50 \le n < 200$ | Lineales + ML | Datos moderados — se prefieren modelos lineales con regularización; los ensambles de árboles pueden funcionar con baja profundidad. Los modelos DL sobreajustarían fuertemente. |
| $200 \le n < 1000$ | Lineales + ML | Suficiente para ML pero aún insuficiente para redes neuronales, que requieren >1000 muestras para una convergencia estable. |
| $n \ge 1000$ | Todas las familias (lineales + ML + DL) | Suficientes datos para que las arquitecturas de deep learning aprendan representaciones significativas sin sobreajuste. |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Restricciones bias-varianza: determinar familias permitidas por experto
# ══════════════════════════════════════════════════════════════════════════════

expert_ids = sorted(df["expert_id"].unique())
n_per_expert = df.groupby("expert_id").size().to_dict()

# Asignar familias de modelos permitidas según el tamaño del clúster
expert_allowed_families = {}
expert_constraints = []

for eid in expert_ids:
    n_obs = n_per_expert[eid]
    if n_obs < 50:
        allowed = ["linear_restricted"]
        constraint = "n < 50: solo Ridge/Lasso (máxima regularización)"
    elif n_obs < 200:
        allowed = ["linear", "ml"]
        constraint = "50 <= n < 200: lineales + ML (sin DL)"
    elif n_obs < 1000:
        allowed = ["linear", "ml"]
        constraint = "200 <= n < 1000: lineales + ML (sin DL)"
    else:
        allowed = ["linear", "ml", "dl"]
        constraint = "n >= 1000: todas las familias"

    expert_allowed_families[eid] = allowed
    expert_constraints.append({
        "expert_id": eid,
        "n_obs": n_obs,
        "allowed_families": ", ".join(allowed),
        "constraint": constraint,
    })

df_constraints = pd.DataFrame(expert_constraints)
print("Restricciones Bias-Varianza por Experto\n")
display(df_constraints)

In [ ]:
def get_models_for_expert(expert_id, family=None):
    """Retorna los modelos permitidos para un experto, opcionalmente filtrados por una sola familia.

    Respeta las restricciones bias-varianza:
      - n < 50:   solo Ridge y Lasso
      - n < 1000: sin DL
    """
    allowed = expert_allowed_families[expert_id]

    models = {}

    if family is None or family == "linear":
        if "linear_restricted" in allowed:
            # Solo Ridge y Lasso para clústeres muy pequeños
            all_linear = get_linear_models()
            models.update({
                k: v for k, v in all_linear.items()
                if k in ["ridge", "lasso"]
            })
        elif "linear" in allowed:
            if family == "linear" or family is None:
                models.update(get_linear_models())

    if family is None or family == "ml":
        if "ml" in allowed:
            models.update(get_ml_models())

    if family is None or family == "dl":
        if "dl" in allowed:
            models.update(get_dl_models())

    return models


# Demostración
for eid in expert_ids:
    m = get_models_for_expert(eid)
    print(f"Experto {eid} (n={n_per_expert[eid]}): {list(m.keys())}")

---

## 2. Evaluación Homogénea (un mismo modelo para TODOS los expertos por familia)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Restricciones bias-varianza: determinar familias permitidas por experto
# ══════════════════════════════════════════════════════════════════════════════

homogeneous_results = []  # lista de diccionarios
homogeneous_best_per_family = {}  # family -> {expert_id: best_model_name}

for family_name in ["linear", "ml", "dl"]:
    print(f"\n{'='*70}")
    print(f"EVALUACIÓN HOMOGÉNEA — Familia: {family_name.upper()}")
    print(f"{'='*70}")

    family_best = {}

    for eid in expert_ids:
        n_obs = n_per_expert[eid]
        mask = df["expert_id"] == eid
        X_expert = df.loc[mask, FEATURE_COLS].values
        y_expert = df.loc[mask, TARGET].values

        # Eliminar filas con NaN
        valid = ~(np.isnan(X_expert).any(axis=1) | np.isnan(y_expert))
        X_expert, y_expert = X_expert[valid], y_expert[valid]

        print(f"\n  Experto {eid} (n={len(X_expert)})")

        # Obtener modelos para esta familia, respetando las restricciones
        if family_name == "linear":
            if "linear_restricted" in expert_allowed_families[eid]:
                all_linear = get_linear_models()
                models = {k: v for k, v in all_linear.items() if k in ["ridge", "lasso"]}
            else:
                models = get_linear_models()
        elif family_name == "ml":
            if "ml" not in expert_allowed_families[eid]:
                print(f"    OMITIDO — ML no permitido para n={n_obs}")
                continue
            models = get_ml_models()
        elif family_name == "dl":
            if "dl" not in expert_allowed_families[eid]:
                print(f"    OMITIDO — DL no permitido para n={n_obs} (se requiere >= 1000)")
                continue
            models = get_dl_models()

        # Evaluar cada modelo de la familia
        best_rmse = float("inf")
        best_model_name = None

        for model_name, model in models.items():
            result = walk_forward_evaluate(model, X_expert, y_expert)
            print(f"    {model_name:25s}  RMSE={result['rmse']:.2f}  MAE={result['mae']:.2f}  MAPE={result['mape']:.4f}")

            homogeneous_results.append({
                "evaluation": "homogeneous",
                "family": family_name,
                "expert_id": eid,
                "n_obs": len(X_expert),
                "model": model_name,
                "rmse": result["rmse"],
                "mae": result["mae"],
                "mape": result["mape"],
                "mase": result["mase"],
            })

            if result["rmse"] < best_rmse:
                best_rmse = result["rmse"]
                best_model_name = model_name

        family_best[eid] = best_model_name
        print(f"    >>> MEJOR: {best_model_name} (RMSE={best_rmse:.2f})")

    homogeneous_best_per_family[family_name] = family_best

print("\n" + "="*70)
print("Evaluación homogénea completada.")
print(f"Total de resultados recopilados: {len(homogeneous_results)}")

In [ ]:
# ── Calcular MASE_global para cada familia homogénea ─────────────────────────

df_homo = pd.DataFrame(homogeneous_results)

# Agregar una verificación para df_homo vacío
if df_homo.empty:
    print("ADVERTENCIA: df_homo está vacío. No hay resultados de evaluación homogénea para procesar.")
    # Inicializar estas variables para evitar errores más adelante si no hay datos
    homo_mase_global = {}
    homo_summary_rows = []
    # Inicializar df_homo_summary como un DataFrame vacío para evitar NameError más adelante
    df_homo_summary = pd.DataFrame(homo_summary_rows)
else:
    homo_mase_global = {}
    homo_summary_rows = []

    for family_name in ["linear", "ml", "dl"]:
        dfam = df_homo[(df_homo["family"] == family_name) & (df_homo["mase"] != np.inf)]
        if dfam.empty:
            continue

        # Mejor modelo por experto dentro de la familia
        best_idx = dfam.groupby("expert_id")["rmse"].idxmin()
        dfam_best = dfam.loc[best_idx]

        mase_per_expert = dict(zip(dfam_best["expert_id"], dfam_best["mase"]))
        n_expert = dict(zip(dfam_best["expert_id"], dfam_best["n_obs"]))

        mase_g = weighted_mase_global(mase_per_expert, n_expert)
        homo_mase_global[family_name] = mase_g

        avg_rmse = dfam_best["rmse"].mean()
        avg_mape = dfam_best["mape"].mean()
        avg_mae = dfam_best["mae"].mean()

        homo_summary_rows.append({
            "family": family_name,
            "n_experts_covered": len(dfam_best),
            "avg_rmse": avg_rmse,
            "avg_mape": avg_mape,
            "avg_mae": avg_mae,
            "mase_global": mase_g,
        })

        print(f"\nFamilia {family_name.upper()}:")
        print(f"  Expertos cubiertos: {len(dfam_best)}")
        print(f"  RMSE promedio:      {avg_rmse:.2f}")
        print(f"  MAPE promedio:      {avg_mape:.4f}")
        print(f"  MAE promedio:       {avg_mae:.4f}")

    df_homo_summary = pd.DataFrame(homo_summary_rows)
print("\n" + "="*50)
print("Resumen de Familias Homogéneas")
display(df_homo_summary.style.format({"avg_rmse": "{:.2f}", "avg_mape": "{:.6f}", "avg_mae": "{:.6f}"}))

---

## 3. Evaluación Heterogénea (mejor modelo por experto entre TODAS las familias)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Evaluación heterogénea — evaluar TODOS los modelos permitidos por experto,
# seleccionar el mejor individual según RMSE
# ══════════════════════════════════════════════════════════════════════════════

heterogeneous_results = []  # lista de diccionarios
hetero_best_models = {}  # expert_id -> (model_name, family, fitted_model)

print("EVALUACIÓN HETEROGÉNEA — Mejor modelo por experto entre todas las familias")
print("=" * 70)

for eid in expert_ids:
    n_obs = n_per_expert[eid]
    mask = df["expert_id"] == eid
    X_expert = df.loc[mask, FEATURE_COLS].values
    y_expert = df.loc[mask, TARGET].values

    # Eliminar filas con NaN
    valid = ~(np.isnan(X_expert).any(axis=1) | np.isnan(y_expert))
    X_expert, y_expert = X_expert[valid], y_expert[valid]

    print(f"\n  Experto {eid} (n={len(X_expert)})")

    # Obtener TODOS los modelos permitidos
    models = get_models_for_expert(eid)

    best_rmse = float("inf")
    best_name = None
    best_family = None

    for model_name, model in models.items():
        # Determinar la familia
        if model_name in get_linear_models():
            fam = "linear"
        elif model_name in get_ml_models():
            fam = "ml"
        else:
            fam = "dl"

        # Verificar si ya tenemos este resultado desde la evaluación homogénea
        existing = [
            r for r in homogeneous_results
            if r["expert_id"] == eid and r["model"] == model_name
        ]
        if existing:
            result = {
                "rmse": existing[0]["rmse"],
                "mae": existing[0]["mae"],
                "mape": existing[0]["mape"],
                "mase": existing[0]["mase"],
            }
        else:
            result = walk_forward_evaluate(model, X_expert, y_expert)

        print(f"    {model_name:25s} [{fam:6s}]  RMSE={result['rmse']:.2f}")

        heterogeneous_results.append({
            "evaluation": "heterogeneous",
            "family": fam,
            "expert_id": eid,
            "n_obs": len(X_expert),
            "model": model_name,
            "rmse": result["rmse"],
            "mae": result["mae"],
            "mape": result["mape"],
            "mase": result["mase"],
        })

        if result["rmse"] < best_rmse:
            best_rmse = result["rmse"]
            best_name = model_name
            best_family = fam

    hetero_best_models[eid] = (best_name, best_family)
    print(f"    >>> MEJOR: {best_name} [{best_family}] (RMSE={best_rmse:.2f})")

print("\n" + "=" * 70)
print("Evaluación heterogénea completada.")

In [ ]:
# ── Tabla de mejores modelos heterogéneos ───────────────────────────────────────────

df_hetero = pd.DataFrame(heterogeneous_results)

# Mejor por experto
best_idx = df_hetero.groupby("expert_id")["rmse"].idxmin()
df_hetero_best = df_hetero.loc[best_idx].sort_values("expert_id").reset_index(drop=True)

print("Selección Heterogénea — Mejor Modelo por Experto")
print("=" * 60)
display(
    df_hetero_best[["expert_id", "model", "family", "rmse", "mape", "mase", "n_obs"]]
    .style.format({"rmse": "{:.2f}", "mape": "{:.6f}", "mase": "{:.6f}"})
)

In [ ]:
# MASE_global para la evaluación heterogénea
df_hetero_best_aux = df_hetero[df_hetero["mase"] != np.inf]
hetero_mase_per_expert = dict(zip(df_hetero_best_aux["expert_id"], df_hetero_best_aux["mase"]))
hetero_n_per_expert = dict(zip(df_hetero_best_aux["expert_id"], df_hetero_best_aux["n_obs"]))
hetero_mase_global = weighted_mase_global(hetero_mase_per_expert, hetero_n_per_expert)

print(f"\nMASE_global heterogéneo = {hetero_mase_global:.4f}")

In [ ]:
# ── Comparar MASE_global heterogéneo vs homogéneo ─────────────────────────

comparison_rows = []
for family_name, mase_g in homo_mase_global.items():
    comparison_rows.append({
        "strategy": f"Homogeneous ({family_name})",
        "mase_global": mase_g,
    })
comparison_rows.append({
    "strategy": "Heterogeneous (best per expert)",
    "mase_global": hetero_mase_global,
})

df_comparison = pd.DataFrame(comparison_rows).sort_values("mase_global")

print("\nComparación de MASE_global: Homogéneo vs Heterogéneo")
print("=" * 55)
display(df_comparison.style.format({"mase_global": "{:.6f}"}).highlight_min(subset=["mase_global"]))

---

## 4. Plots

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 1: Comparación de MASE_global — gráfico de barras
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

strategies = df_comparison["strategy"].values
mase_vals = df_comparison["mase_global"].values
colors = [PALETTE[i] for i in range(len(strategies))]

bars = ax.barh(strategies, mase_vals, color=colors, edgecolor="black", linewidth=0.5)

# Anotar barras
for bar, val in zip(bars, mase_vals):
    ax.text(
        bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", fontsize=10, fontweight="bold",
    )

ax.set_xlabel("MASE global (ponderado por el tamaño del clúster)")
ax.set_title("MASE Global: Selección Homogénea vs. Heterogénea de Expertos")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mase_global_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

strategies = df_comparison["strategy"].values
mase_vals = df_comparison["mase_global"].values
colors = [PALETTE[i] for i in range(len(strategies))]

bars = ax.barh(strategies, mase_vals, color=colors, edgecolor="black", linewidth=0.5)

# Annotate bars
for bar, val in zip(bars, mase_vals):
    ax.text(
        bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", fontsize=10, fontweight="bold",
    )

ax.set_xlabel("MASE global (weighted by cluster size)")
ax.set_title("Global MASE: Homogeneous vs. Heterogeneous Expert Selection")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_mase_global_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 2: RMSE por experto — gráfico de barras agrupadas para familias homogéneas
# ══════════════════════════════════════════════════════════════════════════════

# Obtener el mejor RMSE por experto y por familia a partir de los resultados homogéneos
df_homo_plot = df_homo.copy()
df_homo_best_idx = df_homo_plot.groupby(["family", "expert_id"])["rmse"].idxmin()
df_homo_best = df_homo_plot.loc[df_homo_best_idx]

fig, ax = plt.subplots(figsize=(14, 6))

families_present = df_homo_best["family"].unique()
n_families = len(families_present)
x = np.arange(len(expert_ids))
width = 0.8 / max(n_families, 1)

family_colors = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, fam in enumerate(families_present):
    fam_data = df_homo_best[df_homo_best["family"] == fam]
    rmse_vals = []
    for eid in expert_ids:
        row = fam_data[fam_data["expert_id"] == eid]
        rmse_vals.append(row["rmse"].values[0] if not row.empty else 0)
    offset = (i - n_families / 2 + 0.5) * width
    ax.bar(x + offset, rmse_vals, width, label=fam.upper(), color=family_colors.get(fam, PALETTE[i]),
           edgecolor="black", linewidth=0.4)

# Agregar el mejor heterogéneo como marcadores
hetero_rmse = []
for eid in expert_ids:
    row = df_hetero_best[df_hetero_best["expert_id"] == eid]
    hetero_rmse.append(row["rmse"].values[0] if not row.empty else 0)
ax.scatter(x, hetero_rmse, marker="*", s=120, color="red", zorder=5, label="Mejor heterogéneo")

ax.set_xticks(x)
ax.set_xticklabels(expert_ids, rotation=45, ha="right")
ax.set_xlabel("ID del experto (type_day x type_week)")
ax.set_ylabel("RMSE")
ax.set_title("RMSE por Experto: Familias Homogéneas vs. Mejor Heterogéneo")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_rmse_per_expert.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Plot 2: RMSE per expert — grouped bar chart for homogeneous families
# ══════════════════════════════════════════════════════════════════════════════

# Get best RMSE per expert per family from homogeneous results
df_homo_plot = df_homo.copy()
df_homo_best_idx = df_homo_plot.groupby(["family", "expert_id"])["rmse"].idxmin()
df_homo_best = df_homo_plot.loc[df_homo_best_idx]

fig, ax = plt.subplots(figsize=(14, 6))

families_present = df_homo_best["family"].unique()
n_families = len(families_present)
x = np.arange(len(expert_ids))
width = 0.8 / max(n_families, 1)

family_colors = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, fam in enumerate(families_present):
    fam_data = df_homo_best[df_homo_best["family"] == fam]
    rmse_vals = []
    for eid in expert_ids:
        row = fam_data[fam_data["expert_id"] == eid]
        rmse_vals.append(row["rmse"].values[0] if not row.empty else 0)
    offset = (i - n_families / 2 + 0.5) * width
    ax.bar(x + offset, rmse_vals, width, label=fam.upper(), color=family_colors.get(fam, PALETTE[i]),
           edgecolor="black", linewidth=0.4)

# Add heterogeneous best as markers
hetero_rmse = []
for eid in expert_ids:
    row = df_hetero_best[df_hetero_best["expert_id"] == eid]
    hetero_rmse.append(row["rmse"].values[0] if not row.empty else 0)
ax.scatter(x, hetero_rmse, marker="*", s=120, color="red", zorder=5, label="Heterogeneous best")

ax.set_xticks(x)
ax.set_xticklabels(expert_ids, rotation=45, ha="right")
ax.set_xlabel("Expert ID (type_day x type_week)")
ax.set_ylabel("RMSE")
ax.set_title("RMSE per Expert: Homogeneous Families vs. Heterogeneous Best")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_rmse_per_expert.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 2: MAPE por experto — gráfico de barras agrupadas para familias homogéneas
# ══════════════════════════════════════════════════════════════════════════════

# Obtener el mejor MAPE por experto y por familia a partir de los resultados homogéneos
df_homo_plot = df_homo.copy()
df_homo_best_idx = df_homo_plot.groupby(["family", "expert_id"])["mape"].idxmin()
df_homo_best = df_homo_plot.loc[df_homo_best_idx]

fig, ax = plt.subplots(figsize=(14, 6))

families_present = df_homo_best["family"].unique()
n_families = len(families_present)
x = np.arange(len(expert_ids))
width = 0.8 / max(n_families, 1)

family_colors = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, fam in enumerate(families_present):
    fam_data = df_homo_best[df_homo_best["family"] == fam]
    mape_vals = []
    for eid in expert_ids:
        row = fam_data[fam_data["expert_id"] == eid]
        mape_vals.append(row["mape"].values[0] * 100 if not row.empty else 0)
    offset = (i - n_families / 2 + 0.5) * width
    ax.bar(x + offset, mape_vals, width, label=fam.upper(), color=family_colors.get(fam, PALETTE[i]),
           edgecolor="black", linewidth=0.4)

# Agregar el mejor heterogéneo como marcadores
hetero_mape = []
for eid in expert_ids:
    row = df_hetero_best[df_hetero_best["expert_id"] == eid]
    hetero_mape.append(row["mape"].values[0] * 100 if not row.empty else 0)
ax.scatter(x, hetero_mape, marker="*", s=120, color="red", zorder=5, label="Mejor heterogéneo")

ax.set_xticks(x)
ax.set_xticklabels(expert_ids, rotation=45, ha="right")
ax.set_xlabel("ID del experto (type_day x type_week)")
ax.set_ylabel("MAPE %")
ax.set_title("MAPE por Experto: Familias Homogéneas vs. Mejor Heterogéneo")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mape_per_expert.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Get best MAPE per expert per family from homogeneous results
df_homo_plot = df_homo.copy()
df_homo_best_idx = df_homo_plot.groupby(["family", "expert_id"])["mape"].idxmin()
df_homo_best = df_homo_plot.loc[df_homo_best_idx]

fig, ax = plt.subplots(figsize=(14, 6))

families_present = df_homo_best["family"].unique()
n_families = len(families_present)
x = np.arange(len(expert_ids))
width = 0.8 / max(n_families, 1)

family_colors = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, fam in enumerate(families_present):
    fam_data = df_homo_best[df_homo_best["family"] == fam]
    mape_vals = []
    for eid in expert_ids:
        row = fam_data[fam_data["expert_id"] == eid]
        mape_vals.append(row["mape"].values[0] * 100 if not row.empty else 0)
    offset = (i - n_families / 2 + 0.5) * width
    ax.bar(x + offset, mape_vals, width, label=fam.upper(), color=family_colors.get(fam, PALETTE[i]),
           edgecolor="black", linewidth=0.4)

# Add heterogeneous best as markers
hetero_mape = []
for eid in expert_ids:
    row = df_hetero_best[df_hetero_best["expert_id"] == eid]
    hetero_mape.append(row["mape"].values[0] * 100 if not row.empty else 0)
ax.scatter(x, hetero_mape, marker="*", s=120, color="red", zorder=5, label="Heterogeneous best")

ax.set_xticks(x)
ax.set_xticklabels(expert_ids, rotation=45, ha="right")
ax.set_xlabel("Expert ID (type_day x type_week)")
ax.set_ylabel("MAPE %")
ax.set_title("MAPE per Expert: Homogeneous Families vs. Heterogeneous Best")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_mape_per_expert.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 3: Mapa de calor — RMSE en todos los modelos y expertos
# ══════════════════════════════════════════════════════════════════════════════

# Combinar todos los resultados
all_results = homogeneous_results + [
    r for r in heterogeneous_results
    if not any(
        h["expert_id"] == r["expert_id"] and h["model"] == r["model"]
        for h in homogeneous_results
    )
]
df_all = pd.DataFrame(all_results)

# Pivot para el mapa de calor
pivot = df_all.pivot_table(index="model", columns="expert_id", values="rmse", aggfunc="first")
pivot = pivot.reindex(columns=sorted(pivot.columns))

fig, ax = plt.subplots(figsize=(max(10, len(expert_ids) * 1.2), max(6, len(pivot) * 0.4)))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5,
    ax=ax, cbar_kws={"label": "RMSE"},
)
ax.set_title("Mapa de Calor de RMSE: Modelos (filas) vs. Expertos (columnas)")
ax.set_ylabel("Modelo")
ax.set_xlabel("ID del Experto")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_rmse_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Combine all results
all_results = homogeneous_results + [
    r for r in heterogeneous_results
    if not any(
        h["expert_id"] == r["expert_id"] and h["model"] == r["model"]
        for h in homogeneous_results
    )
]
df_all = pd.DataFrame(all_results)

# Pivot for heatmap
pivot = df_all.pivot_table(index="model", columns="expert_id", values="rmse", aggfunc="first")
pivot = pivot.reindex(columns=sorted(pivot.columns))

fig, ax = plt.subplots(figsize=(max(10, len(expert_ids) * 1.2), max(6, len(pivot) * 0.4)))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5,
    ax=ax, cbar_kws={"label": "RMSE"},
)
ax.set_title("RMSE Heatmap: Models (rows) vs. Experts (columns)")
ax.set_ylabel("Model")
ax.set_xlabel("Expert ID")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_rmse_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 3: Mapa de calor — MAPE en todos los modelos y expertos
# ══════════════════════════════════════════════════════════════════════════════

# Combinar todos los resultados
all_results = homogeneous_results + [
    r for r in heterogeneous_results
    if not any(
        h["expert_id"] == r["expert_id"] and h["model"] == r["model"]
        for h in homogeneous_results
    )
]
df_all = pd.DataFrame(all_results)
df_all["mape"] = df_all["mape"] * 100

# Pivot para el mapa de calor
pivot = df_all.pivot_table(index="model", columns="expert_id", values="mape", aggfunc="first")
pivot = pivot.reindex(columns=sorted(pivot.columns))

fig, ax = plt.subplots(figsize=(max(10, len(expert_ids) * 1.2), max(6, len(pivot) * 0.4)))
sns.heatmap(
    pivot, annot=True, fmt=".3f", cmap="YlOrRd", linewidths=0.5,
    ax=ax, cbar_kws={"label": "MAPE %"},
)
ax.set_title("Mapa de Calor de MAPE: Modelos (filas) vs. Expertos (columnas)")
ax.set_ylabel("Modelo")
ax.set_xlabel("ID del Experto")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mape_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Combine all results
all_results = homogeneous_results + [
    r for r in heterogeneous_results
    if not any(
        h["expert_id"] == r["expert_id"] and h["model"] == r["model"]
        for h in homogeneous_results
    )
]
df_all = pd.DataFrame(all_results)
df_all["mape"] = df_all["mape"] * 100

# Pivot for heatmap
pivot = df_all.pivot_table(index="model", columns="expert_id", values="mape", aggfunc="first")
pivot = pivot.reindex(columns=sorted(pivot.columns))

fig, ax = plt.subplots(figsize=(max(10, len(expert_ids) * 1.2), max(6, len(pivot) * 0.4)))
sns.heatmap(
    pivot, annot=True, fmt=".3f", cmap="YlOrRd", linewidths=0.5,
    ax=ax, cbar_kws={"label": "MAPE %"},
)
ax.set_title("MAPE Heatmap: Models (rows) vs. Experts (columns)")
ax.set_ylabel("Model")
ax.set_xlabel("Expert ID")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_mape_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 4: Tamaño del clúster vs. familia de modelo seleccionada
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 5))

family_marker = {"linear": "o", "ml": "s", "dl": "D"}
family_color = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, eid in enumerate(expert_ids):
    if eid not in hetero_best_models:
        continue
    model_name, fam = hetero_best_models[eid]
    n = n_per_expert[eid]
    ax.scatter(
        n, i, s=120,
        marker=family_marker.get(fam, "o"),
        color=family_color.get(fam, "gray"),
        edgecolor="black", linewidth=0.5, zorder=5,
    )
    ax.annotate(
        f"  {model_name}", (n, i), fontsize=8, va="center",
    )

# Dibujar umbrales de restricción
for threshold, label, ls in [(50, "n=50", "--"), (200, "n=200", "-."), (1000, "n=1000", ":")]:
    ax.axvline(threshold, color="gray", linestyle=ls, alpha=0.5)
    ax.text(threshold + 5, len(expert_ids) - 0.5, label, fontsize=8, color="gray")

ax.set_yticks(range(len(expert_ids)))
ax.set_yticklabels(expert_ids)
ax.set_xlabel("Número de observaciones")
ax.set_ylabel("ID del Experto")
ax.set_title("Selección Heterogénea: Tamaño del Clúster vs. Mejor Familia de Modelo")

# Leyenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker=m, color="w", markerfacecolor=family_color[f],
           markeredgecolor="black", markersize=10, label=f.upper())
    for f, m in family_marker.items()
]
ax.legend(handles=legend_elements, loc="lower right", title="Familia")

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_cluster_size_vs_family.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

family_marker = {"linear": "o", "ml": "s", "dl": "D"}
family_color = {"linear": PALETTE[0], "ml": PALETTE[1], "dl": PALETTE[2]}

for i, eid in enumerate(expert_ids):
    if eid not in hetero_best_models:
        continue
    model_name, fam = hetero_best_models[eid]
    n = n_per_expert[eid]
    ax.scatter(
        n, i, s=120,
        marker=family_marker.get(fam, "o"),
        color=family_color.get(fam, "gray"),
        edgecolor="black", linewidth=0.5, zorder=5,
    )
    ax.annotate(
        f"  {model_name}", (n, i), fontsize=8, va="center",
    )

# Draw constraint thresholds
for threshold, label, ls in [(50, "n=50", "--"), (200, "n=200", "-."), (1000, "n=1000", ":")]:
    ax.axvline(threshold, color="gray", linestyle=ls, alpha=0.5)
    ax.text(threshold + 5, len(expert_ids) - 0.5, label, fontsize=8, color="gray")

ax.set_yticks(range(len(expert_ids)))
ax.set_yticklabels(expert_ids)
ax.set_xlabel("Number of observations")
ax.set_ylabel("Expert ID")
ax.set_title("Heterogeneous Selection: Cluster Size vs. Best Model Family")

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker=m, color="w", markerfacecolor=family_color[f],
           markeredgecolor="black", markersize=10, label=f.upper())
    for f, m in family_marker.items()
]
ax.legend(handles=legend_elements, loc="lower right", title="Family")

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_cluster_size_vs_family.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 5: Diagrama de caja — distribución de RMSE por familia
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

# Mejor RMSE por experto y por familia (desde la evaluación homogénea)
plot_data = []
for fam in ["linear", "ml", "dl"]:
    fam_data = df_homo[df_homo["family"] == fam]
    if fam_data.empty:
        continue
    best_per_expert = fam_data.groupby("expert_id")["rmse"].min()
    for val in best_per_expert.values:
        plot_data.append({"family": fam.upper(), "rmse": val})

df_box = pd.DataFrame(plot_data)

if not df_box.empty:
    sns.boxplot(data=df_box, x="family", y="rmse", palette=[PALETTE[0], PALETTE[1], PALETTE[2]],
                ax=ax, width=0.5)
    sns.stripplot(data=df_box, x="family", y="rmse", color="black", alpha=0.4, size=5, ax=ax)

ax.set_xlabel("Familia de Modelos")
ax.set_ylabel("RMSE (mejor por experto)")
ax.set_title("Distribución de RMSE por Familia (Estrategia Homogénea)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_rmse_boxplot_by_family.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Best RMSE per expert per family (from homogeneous)
plot_data = []
for fam in ["linear", "ml", "dl"]:
    fam_data = df_homo[df_homo["family"] == fam]
    if fam_data.empty:
        continue
    best_per_expert = fam_data.groupby("expert_id")["rmse"].min()
    for val in best_per_expert.values:
        plot_data.append({"family": fam.upper(), "rmse": val})

df_box = pd.DataFrame(plot_data)

if not df_box.empty:
    sns.boxplot(data=df_box, x="family", y="rmse", palette=[PALETTE[0], PALETTE[1], PALETTE[2]],
                ax=ax, width=0.5)
    sns.stripplot(data=df_box, x="family", y="rmse", color="black", alpha=0.4, size=5, ax=ax)

ax.set_xlabel("Model Family")
ax.set_ylabel("RMSE (best per expert)")
ax.set_title("RMSE Distribution by Family (Homogeneous Strategy)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_rmse_boxplot_by_family.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Gráfico 5: Diagrama de caja — distribución de MAPE por familia
# ══════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

# Mejor MAPE por experto y por familia (desde la evaluación homogénea)
plot_data = []
for fam in ["linear", "ml", "dl"]:
    fam_data = df_homo[df_homo["family"] == fam]
    if fam_data.empty:
        continue
    best_per_expert = fam_data.groupby("expert_id")["mape"].min()
    for val in best_per_expert.values:
        plot_data.append({"family": fam.upper(), "mape": val})

df_box = pd.DataFrame(plot_data)

if not df_box.empty:
    sns.boxplot(data=df_box, x="family", y="mape", palette=[PALETTE[0], PALETTE[1], PALETTE[2]],
                ax=ax, width=0.5)
    sns.stripplot(data=df_box, x="family", y="mape", color="black", alpha=0.4, size=5, ax=ax)

ax.set_xlabel("Familia de Modelos")
ax.set_ylabel("MAPE (mejor por experto)")
ax.set_title("Distribución de MAPE por Familia (Estrategia Homogénea)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "05_mape_boxplot_by_family.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Best MAPE per expert per family (from homogeneous)
plot_data = []
for fam in ["linear", "ml", "dl"]:
    fam_data = df_homo[df_homo["family"] == fam]
    if fam_data.empty:
        continue
    best_per_expert = fam_data.groupby("expert_id")["mape"].min()
    for val in best_per_expert.values:
        plot_data.append({"family": fam.upper(), "mape": val})

df_box = pd.DataFrame(plot_data)

if not df_box.empty:
    sns.boxplot(data=df_box, x="family", y="mape", palette=[PALETTE[0], PALETTE[1], PALETTE[2]],
                ax=ax, width=0.5)
    sns.stripplot(data=df_box, x="family", y="mape", color="black", alpha=0.4, size=5, ax=ax)

ax.set_xlabel("Model Family")
ax.set_ylabel("MAPE (best per expert)")
ax.set_title("MAPE Distribution by Family (Homogeneous Strategy)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "english", "05_mape_boxplot_by_family.png"), dpi=300, bbox_inches="tight")
plt.show()

---

## 5. Entrenar Modelos Finales y Guardar

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Entrenar los mejores modelos finales con TODOS los datos por experto y guardar
# ══════════════════════════════════════════════════════════════════════════════

final_models = {}  # expert_id -> modelo ajustado

print("Entrenando modelos finales con todos los datos de cada experto...")
print("=" * 60)

for eid in expert_ids:
    if eid not in hetero_best_models:
        continue

    model_name, family = hetero_best_models[eid]

    # Obtener una nueva instancia del modelo
    all_models = get_models_for_expert(eid)
    if model_name not in all_models:
        print(f"  ADVERTENCIA: {model_name} no se encontró para el experto {eid}, se omitirá.")
        continue

    model = deepcopy(all_models[model_name])

    # Preparar datos
    mask = df["expert_id"] == eid
    X_expert = df.loc[mask, FEATURE_COLS].values
    y_expert = df.loc[mask, TARGET].values

    valid = ~(np.isnan(X_expert).any(axis=1) | np.isnan(y_expert))
    X_expert, y_expert = X_expert[valid], y_expert[valid]

    # Ajustar con todos los datos
    model.fit(X_expert, y_expert)
    final_models[eid] = {
        "model": model,
        "model_name": model_name,
        "family": family,
        "n_obs": len(X_expert),
    }

    print(f"  Experto {eid}: {model_name} [{family}] entrenado con {len(X_expert)} observaciones")

print(f"\nTotal de modelos finales: {len(final_models)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Guardar resultados
# ══════════════════════════════════════════════════════════════════════════════

# 1. CSV completo de resultados
all_results_df = pd.concat([
    pd.DataFrame(homogeneous_results),
    pd.DataFrame(heterogeneous_results),
], ignore_index=True)

csv_path = os.path.join(RESULTS_DIR, "05_expert_selection.csv")
all_results_df.to_csv(csv_path, index=False)
print(f"CSV de resultados guardado: {csv_path}")
print(f"  Total de filas: {len(all_results_df)}")

# 2. Mejores modelos por experto (CSV resumen)
summary_path = os.path.join(RESULTS_DIR, "05_expert_selection_best.csv")
df_hetero_best.to_csv(summary_path, index=False)
print(f"CSV del mejor por experto: {summary_path}")

# 3. Pickle de los modelos finales ajustados
pickle_path = os.path.join(MODELS_DIR, "expert_models.pkl")
with open(pickle_path, "wb") as f:
    pickle.dump(final_models, f)
print(f"Pickle de modelos guardado: {pickle_path}")

# 4. Archivos individuales por experto
experts_dir = os.path.join(MODELS_DIR, "experts")
os.makedirs(experts_dir, exist_ok=True)

for eid, entry in final_models.items():
    # Formato de eid: "day0_week1"
    fname = f"expert_{eid}.pkl"  # -> expert_day0_week1.pkl
    fpath = os.path.join(experts_dir, fname)
    with open(fpath, "wb") as f:
        pickle.dump(entry["model"], f)

print(f"Archivos pkl individuales por experto guardados en: {experts_dir}/")
print(f"  Archivos: {sorted(os.listdir(experts_dir))}")

# 5. Tabla de comparación
comparison_path = os.path.join(RESULTS_DIR, "05_mase_global_comparison.csv")
df_comparison.to_csv(comparison_path, index=False)
print(f"CSV de comparación de MASE: {comparison_path}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Resumen final
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("SELECCIÓN DE EXPERTOS — RESUMEN FINAL")
print("=" * 70)

print(f"\nTotal de expertos:  {len(expert_ids)}")
print(f"Total de modelos evaluados por experto: hasta 15 (5 lineales + 5 ML + 5 DL)")
print(f"Validación cruzada walk-forward: TimeSeriesSplit(n_splits=3, gap=168)")

print("\n--- Restricciones Bias-Varianza ---")
for _, row in df_constraints.iterrows():
    print(f"  Experto {row['expert_id']:10s}  n={row['n_obs']:6d}  -> {row['constraint']}")

print("\n--- Ranking de MASE_global ---")
for _, row in df_comparison.iterrows():
    marker = " <<<" if row["mase_global"] == df_comparison["mase_global"].min() else ""
    print(f"  {row['strategy']:40s}  MASE_global = {row['mase_global']:.4f}{marker}")

print("\n--- Mejor Modelo por Experto (Heterogéneo) ---")
for eid in expert_ids:
    if eid in hetero_best_models:
        name, fam = hetero_best_models[eid]
        row = df_hetero_best[df_hetero_best["expert_id"] == eid].iloc[0]
        print(f"  Experto {eid:10s}  {name:25s} [{fam:6s}]  RMSE={row['rmse']:.2f}  MAPE={row['mape']:.4f}")

print("\n--- Salidas ---")
print(f"  {csv_path}")
print(f"  {summary_path}")
print(f"  {pickle_path}  (diccionario completo, compatibilidad hacia atrás)")
print(f"  {experts_dir}/expert_<eid>.pkl  ({len(final_models)} archivos individuales para NB06+)")
print(f"  {comparison_path}")
print(f"  {os.path.join(RESULTS_DIR, '05_mase_global_comparison.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_rmse_per_expert.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_mape_per_expert.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_rmse_heatmap.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_mape_heatmap.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_cluster_size_vs_family.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_rmse_boxplot_by_family.png')}")
print(f"  {os.path.join(RESULTS_DIR, '05_mape_boxplot_by_family.png')}")
print("\nNotebook 05 completo.")